# Set A V1 Benchmark Tutorial

This notebook shows the practical workflow for comparing and improving Set A V1 solutions without changing the package solver defaults.

The key distinction:

- `scripts/compare_a_v1.py` compares candidate XML files against Hexaly V1 benchmark ratios.
- `scripts/improve_a_v1.py` is a polishing hillclimber. It starts from an existing feasible XML seed; it does not solve from scratch.
- If `--seed` is omitted, the polisher starts from the selected solution listed in the script's `SELECTED_SOLUTIONS` table.
- To continue from an improved run, pass the prior `scratch/a_v1_tuning/<instance>/<instance>_best.xml` as `--seed`.


In [ ]:
from pathlib import Path
import csv
import subprocess
import sys

ROOT = Path.cwd()
DATA = ROOT / "roadef_2016_data"
RESULTS = DATA / "hust_smart_results"
TUNING = ROOT / "scratch" / "a_v1_tuning"

assert (ROOT / "scripts" / "compare_a_v1.py").exists(), ROOT
assert (ROOT / "scripts" / "improve_a_v1.py").exists(), ROOT
assert (DATA / "hexaly_a_benchmarks.csv").exists(), DATA

ROOT


## 1. Inspect the Hexaly V1 benchmark table

Lower logistic ratio is better. These are historical V1 scores, so do not mix them with V2/B/X ratios.

In [ ]:
with (DATA / "hexaly_a_benchmarks.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    print(row["instance"], row["hexaly"], row["best_known"])


## 2. Compare the currently selected local A V1 solutions

This uses the bundled official V1 checker when `mono` and the checker executable are available. The command writes a generated CSV under `hust_smart_results/` by default.

In [ ]:
cmd = [sys.executable, "scripts/compare_a_v1.py"]
print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=False)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0


## 3. Run a bounded polishing pass on one instance

The polisher is deliberately conservative. It tries route reorder, route deletion, source cleanup, and delivery trims. A candidate is accepted only when:

1. local feasibility passes,
2. local ratio improves,
3. official V1 checker validates it,
4. official V1 ratio improves.

The progress counters tell you where time is going.

In [ ]:
instance = "V_1.11"
cmd = [
    sys.executable,
    "scripts/improve_a_v1.py",
    "--instance", instance,
    "--max-passes", "3",
    "--max-reorder-edits", "2000",
    "--max-shift-deletions", "50",
    "--max-source-edits", "100",
    "--max-trim-edits", "300",
]
print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=False)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0


## 4. Resume from an improved seed

Repeated runs without `--seed` start from the selected historical seed again. To continue from an improved result, pass the generated best XML explicitly.

In [ ]:
best_seed = TUNING / instance / f"{instance}_best.xml"
print(best_seed)
print("exists:", best_seed.exists())

if best_seed.exists():
    cmd = [
        sys.executable,
        "scripts/improve_a_v1.py",
        "--instance", instance,
        "--seed", str(best_seed),
        "--max-passes", "3",
        "--max-reorder-edits", "2000",
        "--max-shift-deletions", "50",
        "--max-source-edits", "100",
        "--max-trim-edits", "300",
    ]
    print(" ".join(cmd))
else:
    print("No improved seed yet. Run the previous cell first.")


## 5. Scan tuning output against Hexaly

After polishing, scan `scratch/a_v1_tuning` for the best feasible candidate per instance.

In [ ]:
cmd = [
    sys.executable,
    "scripts/compare_a_v1.py",
    "--scan-candidates",
    "--candidate-dir", str(TUNING),
    "--output-csv", str(TUNING / "comparison.csv"),
]
print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=False)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0


## 6. Solving from scratch versus polishing

The A V1 polisher does not construct a solution from an empty schedule. It requires a feasible seed XML.

To solve from scratch, run one of the package solver entrypoints first, save the output XML, then pass that XML as `--seed` to the polisher. Keep this V1 benchmark lane separate from default solver development so benchmark-specific tuning does not alter package behavior.